# Fitness LLM – Full Pipeline Notebook
### Domain Fine-Tuning + RAG Decision Support System

Implements the pipeline recommended by *Lai et al. (2025)* — "Using Large Language Models to Enhance Exercise Recommendations and Physical Activity" (JMIR Med Inform):

| Phase | Description |
|-------|-------------|
| **1 – Data Preparation** | Load the 2,445-sample fine-tuning dataset built from 7 local data sources |
| **2 – QLoRA Fine-Tuning** | Domain-adapt a Mistral-7B base model on the merged corpus |
| **3 – RAG Index** | Build a FAISS knowledge base from all Q&A pairs |
| **4 – Combined DSS** | Fine-tuned model + RAG retrieval (Figure 3 of the paper) |
| **5 – Evaluation** | Benchmark queries comparing plain model vs. RAG-augmented responses |

---

## 🚀 Recommended Setup — Clone from GitHub (no Drive upload needed)

**All files, including the 2,445-sample training dataset, are already in the GitHub repo.**  
The next code cell handles the clone automatically. You only need to:

### Step 1 — Set your HuggingFace token
In Colab, go to **Tools → Secrets** and add a secret named `HF_TOKEN` with your token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

### Step 2 — (Optional) Set a GitHub token if repo is private
If `purplesquide/fitness-llm` is a **private** repo, add a secret named `GITHUB_TOKEN` with a [Personal Access Token](https://github.com/settings/tokens) that has `repo` read scope.  
If the repo is **public**, skip this step entirely — no token is needed.

### Step 3 — Run cells in order
- **Cell 2** clones the repo into `/content/fitness-llm/` and sets the working directory.
- **Cell 3** installs dependencies and **automatically restarts the runtime** when done (this is required to flush stale package versions — numpy and pandas in particular).
- After the restart, **continue from Cell 4** (HuggingFace login). You can skip Cells 2 and 3 — the repo is already cloned and packages are installed.

---

## ☁️ Alternative Setup — Google Drive (if you prefer not to use GitHub)

If you want to avoid cloning and use Drive instead:
1. Set `USE_GITHUB = False` in the **second cell** (the Setup cell)
2. Upload the repo folder to `MyDrive/fitness_llm_colab/` containing at minimum:
   - `requirements-colab.txt`
   - `pyproject.toml`  
   - `README.md`
   - `artifacts/datasets/fitness_training_data_full.jsonl`

---

> **Runtime requirement**: GPU (T4 or better). Set `HF_TOKEN` in Colab Secrets before running.

In [ ]:
import os
import subprocess
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
# SETUP — choose your source
# ─────────────────────────────────────────────────────────────────────────────
USE_GITHUB = True   # True  → clone from GitHub (recommended, no Drive needed)
                    # False → load from Google Drive

GITHUB_REPO = 'purplesquide/fitness-llm'   # change if you forked the repo
CLONE_DIR   = '/content/fitness-llm'       # where the repo lands in Colab
# ─────────────────────────────────────────────────────────────────────────────


REQUIRED_FILES = ('requirements-colab.txt', 'pyproject.toml', 'README.md')


def looks_like_project_root(path: Path) -> bool:
    return path.is_dir() and all((path / name).exists() for name in REQUIRED_FILES)


# ── Path A: GitHub clone ──────────────────────────────────────────────────────
if USE_GITHUB:
    clone_path = Path(CLONE_DIR)

    if looks_like_project_root(clone_path):
        print(f'Repo already cloned at {clone_path}')
    else:
        print(f'Cloning {GITHUB_REPO} → {CLONE_DIR} ...')

        # Build the clone URL — support private repos via GITHUB_TOKEN secret
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
            clone_url = f'https://{token}@github.com/{GITHUB_REPO}.git'
            print('  (using GITHUB_TOKEN for private repo)')
        except Exception:
            clone_url = f'https://github.com/{GITHUB_REPO}.git'
            print('  (public repo — no token needed)')

        result = subprocess.run(
            ['git', 'clone', '--depth', '1', clone_url, CLONE_DIR],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            raise RuntimeError(
                f'git clone failed:\n{result.stderr}\n\n'
                'If this is a private repo, add your Personal Access Token as a '
                'Colab Secret named GITHUB_TOKEN (Tools → Secrets).'
            )
        print(result.stdout or 'Clone complete.')

    PROJECT_ROOT = clone_path

# ── Path B: Google Drive ──────────────────────────────────────────────────────
else:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted.')

    DRIVE_CANDIDATES = [
        Path('/content/drive/MyDrive/fitness_llm_colab'),
        Path('/content/drive/MyDrive/fitness-llm'),
    ]

    PROJECT_ROOT = next(
        (p for p in DRIVE_CANDIDATES if looks_like_project_root(p)),
        None
    )

    if PROJECT_ROOT is None:
        # Fallback: broad search up to 3 levels deep
        for candidate in Path('/content/drive/MyDrive').rglob('*'):
            if len(candidate.parts) - len(Path('/content/drive/MyDrive').parts) > 3:
                continue
            if looks_like_project_root(candidate):
                PROJECT_ROOT = candidate
                break

    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            'Project root not found on Drive.\n'
            'Make sure you uploaded the repo to MyDrive/fitness_llm_colab/ '
            'with requirements-colab.txt, pyproject.toml, and README.md present.'
        )

# ── Set working directory for all subsequent cells ───────────────────────────
os.chdir(PROJECT_ROOT)
print(f'\n✓ Working directory : {PROJECT_ROOT.resolve()}')
print(f'✓ Dataset present   : {(PROJECT_ROOT / "artifacts" / "datasets" / "fitness_training_data_full.jsonl").exists()}')

In [ ]:
# Force-upgrade packages that Colab pre-installs at incompatible versions.
# MUST run before requirements.txt because `pip install -r` skips already-installed packages.
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

# bitsandbytes >=0.45.0 — 0.44.x has broken cpu backend (has_avx512bf16 missing)
pip('--upgrade', '--force-reinstall', 'bitsandbytes>=0.45.0')

# trl >=0.12.0 — 0.10.x uses tokenizer=, 0.12+ uses processing_class= (required by this notebook)
pip('--upgrade', '--force-reinstall', 'trl>=0.12.0')

# pandas >=2.2.0 — 2.1.x was built for numpy 1.x ABI; 2.2+ supports numpy 2.x
pip('--upgrade', 'pandas>=2.2.0')

# Install all remaining project requirements
pip('-r', 'requirements-colab.txt')
pip('-e', '.')

print('✓ Installation complete — restarting runtime to load new package versions...')
print('  After the restart, continue from Cell 4 (HuggingFace login).')

import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

In [ ]:
from huggingface_hub import login
import os

HF_TOKEN = None

# 1. Try Colab Secrets (Tools → Secrets → HF_TOKEN, Notebook access: ON)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    if HF_TOKEN:
        print('✓ HF_TOKEN loaded from Colab Secrets.')
    else:
        print('⚠ HF_TOKEN secret exists but is empty.')
except ImportError:
    pass   # not running in Colab
except Exception as e:
    print(f'⚠ Could not read Colab Secret: {e}')
    print('  → Make sure "Notebook access" is ON for HF_TOKEN in the Secrets panel.')

# 2. Fall back to environment variable (for local / non-Colab runs)
if not HF_TOKEN:
    HF_TOKEN = os.getenv('HF_TOKEN', '')
    if HF_TOKEN:
        print('✓ HF_TOKEN loaded from environment variable.')

if HF_TOKEN:
    login(HF_TOKEN, add_to_git_credential=False)
    print('✓ Logged in to HuggingFace.')
else:
    raise EnvironmentError(
        '\nHF_TOKEN not found. Steps to fix:\n'
        '  1. Get a token: https://huggingface.co/settings/tokens (type: Read)\n'
        '  2. In Colab: click 🔑 in the left sidebar → Add new secret\n'
        '     Name: HF_TOKEN\n'
        '     Value: hf_...\n'
        '     *** Toggle "Notebook access" to ON *** ← most common miss\n'
        '  3. Re-run this cell (no restart needed).'
    )

---
## Phase 1 – Data Preparation

The paper stresses using **diverse, high-quality domain data** — exercise guidance, program advice, nutrition, recovery, and app-specific concepts.

The primary dataset is `fitness_training_data_full.jsonl` — built by the `fitness_dataset/` pipeline from 7 local data sources:

| File | Samples | Content |
|------|---------|---------|
| `fitness_training_data_full.jsonl` | **2,445** | ✅ Primary — exercise guidance, substitutions, programs, nutrition, injury, app concepts |
| `nutrition_textbook_full_qa.jsonl` | 200 | Nutrition textbook Q&A (already included in full dataset above after dedup) |
| `full_llm_training_data.jsonl` | 306 | Prior merged corpus (already included above after dedup) |

The cell below loads `fitness_training_data_full.jsonl` as the primary source, then deduplicates against any optional extras.

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

ROOT = Path('.')

# ── Dataset registry ──────────────────────────────────────────────────────────
# fitness_training_data_full.jsonl is the PRIMARY dataset (2,445 samples built
# from ExerciseDB, MegaGym, wger, Longhaul, HuggingFace fitness QA, and manual
# high-quality samples covering nutrition, progress, injury, RPE, and habits).
# The remaining files are optional extras — they are deduped on load, so
# uploading them adds no new samples if you already have the full dataset.
DATASET_FILENAMES = {
    'fitness_full':              'fitness_training_data_full.jsonl',   # PRIMARY (2,445 samples)
    'nutrition_textbook_full_qa':'nutrition_textbook_full_qa.jsonl',   # optional extra
    'full_llm_corpus':           'full_llm_training_data.jsonl',       # optional extra
}

DATASET_DIR_CANDIDATES = [
    ROOT,
    ROOT / 'artifacts' / 'datasets',
]

DATASET_DIR = next(
    (
        candidate for candidate in DATASET_DIR_CANDIDATES
        if candidate.exists() and (candidate / 'fitness_training_data_full.jsonl').exists()
    ),
    None,
)

if DATASET_DIR is None:
    raise FileNotFoundError(
        'fitness_training_data_full.jsonl not found.\n'
        'Expected location: artifacts/datasets/fitness_training_data_full.jsonl\n'
        'Make sure you uploaded this file to Google Drive under fitness_llm_colab/artifacts/datasets/'
    )

print(f'Dataset directory : {DATASET_DIR.resolve()}')
print(f'Primary dataset   : {(DATASET_DIR / "fitness_training_data_full.jsonl").stat().st_size / 1024:.0f} KB')

DATASET_FILES = {
    key: DATASET_DIR / filename
    for key, filename in DATASET_FILENAMES.items()
}


def load_jsonl(path: Path) -> list[dict]:
    samples = []
    with path.open(encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    samples.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return samples


# ── Load and deduplicate ──────────────────────────────────────────────────────
# Load PRIMARY dataset first, then append optional extras (deduped by user message).
all_samples: list[dict] = []
seen_keys: set[str] = set()
source_counts: dict[str, int] = defaultdict(int)

for source_name, path in DATASET_FILES.items():
    if not path.exists():
        status = '⚠ MISSING (optional)' if source_name != 'fitness_full' else '✗ MISSING — required!'
        print(f'  [{status}] {path.name}')
        if source_name == 'fitness_full':
            raise FileNotFoundError(f'Primary dataset not found: {path}')
        continue

    raw = load_jsonl(path)
    added = 0
    for sample in raw:
        msgs = sample.get('messages', [])
        user_msgs = [m['content'] for m in msgs if m.get('role') == 'user']
        key = '|'.join(user_msgs).strip().lower()[:200]
        if key and key not in seen_keys:
            seen_keys.add(key)
            all_samples.append(sample)
            added += 1
    source_counts[source_name] = added
    print(f'  {path.name:50s}  +{added:4d} unique  (running total: {len(all_samples)})')

print(f'\n{"─" * 60}')
print(f'  Total unique training samples : {len(all_samples)}')
print(f'{"─" * 60}')

if not all_samples:
    raise ValueError('No valid samples were loaded. Check that the JSONL files contain chat records.')

# Write merged dataset to root for downstream training cells
MERGED_PATH = ROOT / 'all_datasets_merged.jsonl'
with MERGED_PATH.open('w', encoding='utf-8') as f:
    for s in all_samples:
        f.write(json.dumps(s, ensure_ascii=False) + '\n')
print(f'\nMerged dataset saved → {MERGED_PATH.resolve()}')

# Preview
print('\n── Sample (first entry) ──')
s = all_samples[0]
print('User   :', s['messages'][1]['content'][:120])
print('Answer :', s['messages'][2]['content'][:120], '...')

In [ ]:
# Dataset quality checks & topic distribution
import re

def classify_topic(sample: dict) -> str:
    user_text = ' '.join(
        m['content'] for m in sample.get('messages', []) if m.get('role') == 'user'
    ).lower()
    if any(w in user_text for w in ['protein', 'calorie', 'carb', 'fat', 'vitamin', 'nutrient', 'diet', 'food', 'eat', 'meal', 'nutrition']):
        return 'Nutrition'
    if any(w in user_text for w in ['workout', 'exercise', 'training', 'muscle', 'strength', 'cardio', 'rep', 'set', 'lift']):
        return 'Exercise'
    if any(w in user_text for w in ['weight loss', 'bmi', 'body fat', 'obesity', 'slim', 'bulk', 'cut']):
        return 'Body Composition'
    if any(w in user_text for w in ['sleep', 'recover', 'rest', 'injury', 'pain', 'rehabilitation']):
        return 'Recovery'
    return 'General Health'

topic_counts: dict[str, int] = defaultdict(int)
token_lens: list[int] = []

for s in all_samples:
    topic_counts[classify_topic(s)] += 1
    total_text = ' '.join(m['content'] for m in s.get('messages', []))
    token_lens.append(len(total_text.split()))

print('Topic distribution')
print('─' * 40)
for topic, count in sorted(topic_counts.items(), key=lambda x: -x[1]):
    bar = '█' * (count * 30 // max(topic_counts.values()))
    print(f'  {topic:<20s} {count:4d}  {bar}')

print(f'\nApproximate token stats (whitespace-split words)')
print(f'  Min  : {min(token_lens)}')
print(f'  Max  : {max(token_lens)}')
print(f'  Mean : {sum(token_lens) / len(token_lens):.0f}')
print(f'  >512 tokens: {sum(1 for l in token_lens if l > 512)} samples')


---
## Phase 2 – QLoRA Domain Fine-Tuning

Following the paper's recommendation to **fine-tune existing LLMs with domain-specific corpora** (exercise science literature, Q&A, clinical guidelines).  
We use **4-bit QLoRA** (quantised Low-Rank Adaptation) to fit Mistral-7B on a single GPU:

- **Base model**: `mistralai/Mistral-7B-Instruct-v0.3`  
- **LoRA rank**: 16, alpha 32 — targets all attention + MLP projection layers  
- **Dataset**: merged corpus (`all_datasets_merged.jsonl`)  
- **Epochs**: 3, batch 2 × grad-accum 4 → effective batch 8

In [ ]:
import os
import torch
import trl
from datasets import load_dataset
from packaging.version import Version
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime is required for training. Runtime → Change runtime type → T4 GPU')

BASE_MODEL_ID = os.getenv('BASE_MODEL_ID', 'mistralai/Mistral-7B-Instruct-v0.3')
ADAPTER_DIR   = 'fitness-mistral-qlora-final'
MAX_SEQ_LEN   = 1024

major, _ = torch.cuda.get_device_capability()
use_bf16  = major >= 8   # Ampere+ (A100, H100) supports bfloat16 natively
use_fp16  = not use_bf16
dtype     = torch.bfloat16 if use_bf16 else torch.float16
trl_ver   = Version(trl.__version__)
print(f'GPU: {torch.cuda.get_device_name(0)}  |  dtype: {"bfloat16" if use_bf16 else "float16"}')
print(f'TRL version: {trl.__version__}')

# paged_adamw_8bit stores internal states in bfloat16 in newer bitsandbytes.
# On fp16 GPUs (T4, V100) the AMP grad scaler can't unscale bfloat16 tensors
# → NotImplementedError. Use paged_adamw_32bit for fp16 GPUs instead.
optim = 'paged_adamw_8bit' if use_bf16 else 'paged_adamw_32bit'
print(f'Optimizer: {optim}')

# 4-bit quantisation config
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

# max_seq_length API changed across TRL versions:
#   TRL <0.13  → SFTTrainer(max_seq_length=...)
#   TRL 0.13.x → SFTConfig(max_seq_length=...)
#   TRL >=1.0  → removed; set tokenizer.model_max_length instead
if trl_ver >= Version('1.0.0'):
    tokenizer.model_max_length = MAX_SEQ_LEN
    sft_config_extra  = {}
    sft_trainer_extra = {}
elif trl_ver >= Version('0.13.0'):
    sft_config_extra  = {'max_seq_length': MAX_SEQ_LEN}
    sft_trainer_extra = {}
else:
    sft_config_extra  = {}
    sft_trainer_extra = {'max_seq_length': MAX_SEQ_LEN}

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True,
    attn_implementation='eager',
    dtype=dtype,
)
model = prepare_model_for_kbit_training(model)

# After prepare_model_for_kbit_training, some parameters (layer norms, embeddings)
# may still be bfloat16 from the model's own config.json, even though we loaded
# with dtype=float16. Any bf16 param produces bf16 gradients → the fp16 AMP grad
# scaler raises NotImplementedError. Explicitly cast all remaining bf16 params.
if not use_bf16:
    bf16_count = 0
    for param in model.parameters():
        if param.dtype == torch.bfloat16:
            param.data = param.data.to(torch.float16)
            bf16_count += 1
    if bf16_count:
        print(f'Cast {bf16_count} bf16 parameter tensors → fp16')

# NOTE: do NOT call get_peft_model() here — SFTTrainer applies LoRA itself
# when peft_config is passed (TRL 1.x behaviour).

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

# Load dataset
dataset = load_dataset('json', data_files='all_datasets_merged.jsonl', split='train')
split   = dataset.train_test_split(test_size=0.1, seed=42)

def fmt(example):
    return {'text': tokenizer.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False
    )}

train_ds = split['train'].map(fmt)
eval_ds  = split['test'].map(fmt)
print(f'Train: {len(train_ds)}  |  Eval: {len(eval_ds)}')

args = SFTConfig(
    output_dir=ADAPTER_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    bf16=use_bf16,
    fp16=use_fp16,
    logging_steps=10,
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    eval_strategy='steps',
    load_best_model_at_end=True,
    optim=optim,
    dataset_text_field='text',
    report_to='none',
    **sft_config_extra,
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=lora,
    **sft_trainer_extra,
)
trainer.model.print_trainable_parameters()
trainer.train()
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'\nAdapter saved → {ADAPTER_DIR}')


---
## Phase 3 – RAG Knowledge Base

The paper highlights **Retrieval-Augmented Generation (RAG)** as a key direction: it "ensures any knowledge gaps in the language model are promptly addressed, reducing the risk of misinformation and enhancing the accuracy of exercise guidance."

We build a FAISS vector index from **all** Q&A pairs across every dataset:
- Each Q&A pair becomes a retrievable knowledge chunk: `Q: … A: …`
- Embeddings: `sentence-transformers/all-MiniLM-L6-v2`
- At inference time, the top-*k* chunks are injected into the system prompt before generation

In [ ]:
import json
import pickle
import numpy as np
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer

ROOT = Path('.')
INDEX_PATH = ROOT / 'fitness_knowledge.index'
CHUNKS_PATH = ROOT / 'fitness_knowledge_chunks.pkl'
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

# Reuse the dataset paths discovered in Phase 1 when available.
if 'DATASET_FILES' in globals():
    SOURCE_DATASET_PATHS = [path for path in DATASET_FILES.values() if path.exists()]
else:
    fallback_dir = ROOT / 'artifacts' / 'datasets'
    SOURCE_DATASET_PATHS = sorted(fallback_dir.glob('*.jsonl')) if fallback_dir.exists() else sorted(ROOT.glob('*.jsonl'))

if 'MERGED_PATH' in globals() and Path(MERGED_PATH).exists():
    SOURCE_DATASET_PATHS.append(Path(MERGED_PATH))

# De-duplicate paths while preserving order.
unique_paths = []
seen_paths = set()
for path in SOURCE_DATASET_PATHS:
    resolved = str(Path(path))
    if resolved not in seen_paths:
        seen_paths.add(resolved)
        unique_paths.append(Path(path))
SOURCE_DATASET_PATHS = unique_paths

if not SOURCE_DATASET_PATHS:
    raise FileNotFoundError('No dataset files available for RAG indexing.')

print('RAG sources:')
for path in SOURCE_DATASET_PATHS:
    print(f'  - {path}')

# Build knowledge chunks from Q&A pairs as: "Q: ...\nA: ..."
def qa_to_chunks(path: Path) -> list[dict]:
    chunks = []
    with path.open(encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                sample = json.loads(line)
            except json.JSONDecodeError:
                continue
            msgs = sample.get('messages', [])
            user = next((m['content'] for m in msgs if m['role'] == 'user'), None)
            asst = next((m['content'] for m in msgs if m['role'] == 'assistant'), None)
            if user and asst:
                chunk_text = f"Q: {user.strip()}\nA: {asst.strip()}"
                chunks.append({'source': path.stem, 'text': chunk_text})
    return chunks

all_chunks: list[dict] = []
seen_texts: set[str] = set()
for ds_path in SOURCE_DATASET_PATHS:
    raw = qa_to_chunks(ds_path)
    added = 0
    for chunk in raw:
        key = chunk['text'][:200]
        if key not in seen_texts:
            seen_texts.add(key)
            all_chunks.append(chunk)
            added += 1
    print(f'  {ds_path.name:45s}  +{added:4d} chunks  (total: {len(all_chunks)})')

print(f'\nTotal knowledge chunks: {len(all_chunks)}')

embedder = SentenceTransformer(EMBED_MODEL)
texts = [chunk['text'] for chunk in all_chunks]
embeddings = embedder.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
).astype(np.float32)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
faiss.write_index(index, str(INDEX_PATH))
with CHUNKS_PATH.open('wb') as fh:
    pickle.dump(all_chunks, fh)

print(f'\nFAISS index saved   -> {INDEX_PATH.resolve()}')
print(f'Chunks pickle saved -> {CHUNKS_PATH.resolve()}')
print(f'Index size: {index.ntotal} vectors, dim {embeddings.shape[1]}')


In [ ]:
# Test the retrieval system with sample queries
import pickle, faiss, numpy as np
from sentence_transformers import SentenceTransformer
from pathlib import Path

def retrieve(query: str, top_k: int = 3) -> list[dict]:
    idx   = faiss.read_index(str(INDEX_PATH))
    with CHUNKS_PATH.open('rb') as fh:
        chunks = pickle.load(fh)
    emb = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, ids = idx.search(emb, top_k)
    results = []
    for score, i in zip(scores[0], ids[0]):
        if i >= 0:
            results.append({'score': float(score), **chunks[i]})
    return results

test_queries = [
    'How much protein should I eat per day to build muscle?',
    'What is the best cardio workout for weight loss?',
    'How do carbohydrates affect athletic performance?',
    'What are the signs of overtraining?',
]

for q in test_queries:
    print(f'\n🔍 Query: {q}')
    for r in retrieve(q, top_k=2):
        preview = r['text'].replace('\n', ' ')[:120]
        print(f'  [{r["score"]:.3f}] ({r["source"]}) {preview} …')


---
## Phase 4 – Combined DSS: Fine-Tuned Model + RAG

This implements the **Decision Support System** described in Figure 3 of the paper:
> *"Advanced LLMs integrated within a DSS will use retrieval-augmented generation and fine-tuning techniques based on extensive corpora."*

**Inference flow:**
1. Receive user query
2. Retrieve top-*k* relevant knowledge chunks via FAISS (RAG)
3. Inject retrieved context into the system prompt
4. Generate response using the fine-tuned Mistral-7B adapter

The RAG step ensures the model always has access to the most relevant domain knowledge, compensating for any gaps introduced by fine-tuning on a limited corpus.

In [ ]:
import torch
import faiss, pickle, numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from sentence_transformers import SentenceTransformer

ADAPTER_DIR   = 'fitness-mistral-qlora-final'   # root directory
BASE_MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'

# ── Load fine-tuned model ──────────────────────────────────────────────────
major, _ = torch.cuda.get_device_capability()
dtype = torch.bfloat16 if major >= 8 else torch.float16

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=dtype,
    bnb_4bit_use_double_quant=True,
)

print('Loading base model …')
chat_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True,
    attn_implementation='eager',
)
chat_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
chat_model.eval()
print('Fine-tuned model ready.')

# ── Load RAG components (root directory) ──────────────────────────────────
rag_index  = faiss.read_index('fitness_knowledge.index')
with open('fitness_knowledge_chunks.pkl', 'rb') as fh:
    rag_chunks = pickle.load(fh)
rag_embedder = SentenceTransformer(EMBED_MODEL)
print(f'RAG index loaded: {rag_index.ntotal} chunks')

# ── Retrieval helper ───────────────────────────────────────────────────────
def rag_retrieve(query: str, top_k: int = 3) -> str:
    emb = rag_embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, ids = rag_index.search(emb, top_k)
    context_parts = []
    for score, i in zip(scores[0], ids[0]):
        if i >= 0 and score > 0.25:   # similarity threshold (paper: mitigate misinformation)
            context_parts.append(rag_chunks[i]['text'])
    return '\n\n'.join(context_parts)

# ── Generation helper ──────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are a knowledgeable fitness and nutrition coach. "
    "Provide evidence-based, personalised exercise and nutrition guidance. "
    "Always emphasise safety and recommend consulting a healthcare professional "
    "for medical conditions."
)

def chat(user_query: str, use_rag: bool = True, max_new_tokens: int = 300) -> str:
    system = SYSTEM_PROMPT
    if use_rag:
        context = rag_retrieve(user_query)
        if context:
            system += f"\n\nRelevant knowledge:\n{context}"

    messages = [
        {'role': 'system', 'content': system},
        {'role': 'user',   'content': user_query},
    ]
    input_ids = chat_tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to(chat_model.device)

    with torch.no_grad():
        out = chat_model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=chat_tokenizer.eos_token_id,
        )
    response = chat_tokenizer.decode(
        out[0][input_ids.shape[1]:], skip_special_tokens=True
    )
    return response.strip()

print('DSS pipeline ready.  Call chat("your question") to query the model.')


In [ ]:
# ── Interactive single-turn chat cell ─────────────────────────────────────
# Edit the question below and re-run this cell at any time.
QUESTION = "I'm a 30-year-old beginner who wants to lose weight. What exercise plan and diet should I follow?"

print('USER:', QUESTION)
print()
answer = chat(QUESTION, use_rag=True)
print('ASSISTANT (RAG + fine-tuned model):')
print(answer)


---
## Phase 5 – Evaluation

The paper recommends validating LLM outputs for **accuracy, safety, and personalisation** (usability testing via expert evaluation and user interaction metrics).

We run a small benchmark that:
1. Tests the model on **5 representative queries** covering all topic categories
2. Compares responses **with and without RAG** to demonstrate retrieval benefit
3. Reports response length as a proxy for completeness

> In a production setting, qualified exercise physiologists should review generated plans before clinical application, as stressed throughout the paper.

In [ ]:
BENCHMARK_QUERIES = [
    # (topic, query)
    ('Exercise',         'Design a 4-week beginner strength training programme.'),
    ('Nutrition',        'How much protein, carbs, and fat should an endurance athlete eat?'),
    ('Body Composition', 'What combination of diet and exercise is most effective for fat loss while preserving muscle?'),
    ('Recovery',         'How should I structure rest and recovery days in my weekly training plan?'),
    ('General Health',   'What are the ACSM guidelines for physical activity for adults?'),
]

results = []
print('Running benchmark (with vs without RAG) …\n')
for topic, query in BENCHMARK_QUERIES:
    ans_rag  = chat(query, use_rag=True,  max_new_tokens=250)
    ans_base = chat(query, use_rag=False, max_new_tokens=250)
    results.append({
        'topic':      topic,
        'query':      query,
        'rag_words':  len(ans_rag.split()),
        'base_words': len(ans_base.split()),
        'rag_answer': ans_rag,
    })
    print(f'[{topic}]')
    print(f'  Q: {query}')
    print(f'  RAG answer  ({len(ans_rag.split())} words): {ans_rag[:200]} …')
    print(f'  Base answer ({len(ans_base.split())} words): {ans_base[:200]} …')
    print()

# Summary table
print('─' * 60)
print(f'{"Topic":<22} {"RAG words":>10} {"Base words":>11}')
print('─' * 60)
for r in results:
    diff = r['rag_words'] - r['base_words']
    diff_str = f'+{diff}' if diff >= 0 else str(diff)
    print(f'{r["topic"]:<22} {r["rag_words"]:>10} {r["base_words"]:>11}   ({diff_str})')
print('─' * 60)
print('\nPositive diff → RAG produced more detailed response.')


---
## Summary & Next Steps

### What was built (aligned with Lai et al. 2025)

| Paper Recommendation | Implementation |
|---|---|
| Diverse domain-specific corpora | 6 dataset files merged (handcrafted Q&A + full fitness + nutrition textbook) |
| Fine-tuning on domain data | QLoRA fine-tuning of Mistral-7B-Instruct on merged corpus |
| RAG to fill knowledge gaps | FAISS index from all Q&A pairs; top-k chunks injected at inference time |
| Combined DSS (fine-tune + RAG) | `chat()` function: retrieves context → fine-tuned model generates response |
| Safety & expert validation | System prompt includes disclaimer; expert review recommended before clinical use |

### Suggested next steps (from the paper's Future Directions)
- **Real-world clinical trials**: collect practitioner feedback and refine outputs
- **Wearable integration**: feed real-time heart rate / activity data as additional context
- **Reinforcement learning from expert feedback**: expert corrections → RLHF/DPO fine-tuning loop
- **Standardised evaluation metrics**: ACSM-guideline alignment score, safety screening

> *"With these advancements, LLMs can evolve into a valuable decision support tool, enhancing accessibility and personalization in exercise science while maintaining expert oversight."* — Lai et al., 2025